In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Query local DataFrames with SQL

https://www.ers.usda.gov/data-products/wheat-data


In [2]:
%pip install python-calamine pandas bigframes

  Using cached pandas-2.3.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
Using cached pandas-2.3.3-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.3 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.2
    Uninstalling pandas-3.0.2:
      Successfully uninstalled pandas-3.0.2
Note: you may need to restart the kernel to use updated packages.


In [3]:
!wget -O /tmp/wheat-data.xlsx 'https://www.ers.usda.gov/media/5706/wheat-data-all-years.xlsx?v=19753'

--2026-05-07 17:55:50--  https://www.ers.usda.gov/media/5706/wheat-data-all-years.xlsx?v=19753
Resolving www.ers.usda.gov (www.ers.usda.gov)... 20.141.137.224
Connecting to www.ers.usda.gov (www.ers.usda.gov)|20.141.137.224|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 814596 (796K) [application/vnd.openxmlformats-officedocument.spreadsheetml.sheet]
Saving to: ‘/tmp/wheat-data.xlsx’

/tmp/wheat-data.xls 100%[===================>] 795.50K  2.32MB/s    in 0.3s    

2026-05-07 17:55:51 (2.32 MB/s) - ‘/tmp/wheat-data.xlsx’ saved [814596/814596]



In [4]:
import pandas as pd

df = pd.read_excel(
    "/tmp/wheat-data.xlsx",
    sheet_name="Table06",
    header=1,

    # Requires that the python-calamine project is also installed.
    engine="calamine",

    # Recommended so that string columns don't contain NaN, which can confuse
    # parquet serialization, which we use to read these data in BigQuery SQL.
    dtype_backend="pyarrow",
)
df

,Marketing year 1/,Type 2/,Beginning stocks,Production,Imports,Total supply 3/ 4/,Food use,Seed use,Feed and residual use,Total domestic use 4/,Exports,Total disappearance 4/,Ending stocks
0,1950/51,All wheat,496.0,1019.0,11,1526.0,580.0,--,109.0,689.0,345.0,1034.0,492.0
1,1951/52,All wheat,492.0,988.0,30,1510.0,585.0,--,110.0,695.0,485.0,1180.0,330.0
2,1952/53,All wheat,330.0,1306.0,24,1660.0,578.0,--,78.0,656.0,332.0,988.0,672.0
3,1953/54,All wheat,672.0,1173.0,6,1851.0,556.0,--,87.0,643.0,214.0,857.0,994.0
4,1954/55,All wheat,994.0,984.0,3,1981.0,552.0,--,53.0,605.0,267.0,872.0,1109.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,"2/ Hard Red Winter, Hard Red Spring, Soft Red ...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
288,3/ Includes flour and selected other products ...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
289,4/ Totals may not add due to rounding.,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
290,"Source: USDA, Economic Research Service, based...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [5]:
# To aid in SQL authoring, rename the columns to avoid problematic special
# characters.  Note: BigQuery supports some special characters, but not "/".
# https://docs.cloud.google.com/bigquery/docs/schemas#flexible-column-names
df_renamed = df.rename(
    columns={
        column: column.replace("/", "")
        for column in df.columns
    }
)

# Also, give a name to the index so that it can be included.
df_renamed.index.name = "rowindex"
df_renamed


,Marketing year 1,Type 2,Beginning stocks,Production,Imports,Total supply 3 4,Food use,Seed use,Feed and residual use,Total domestic use 4,Exports,Total disappearance 4,Ending stocks
rowindex,,,,,,,,,,,,,
0,1950/51,All wheat,496.0,1019.0,11,1526.0,580.0,--,109.0,689.0,345.0,1034.0,492.0
1,1951/52,All wheat,492.0,988.0,30,1510.0,585.0,--,110.0,695.0,485.0,1180.0,330.0
2,1952/53,All wheat,330.0,1306.0,24,1660.0,578.0,--,78.0,656.0,332.0,988.0,672.0
3,1953/54,All wheat,672.0,1173.0,6,1851.0,556.0,--,87.0,643.0,214.0,857.0,994.0
4,1954/55,All wheat,994.0,984.0,3,1981.0,552.0,--,53.0,605.0,267.0,872.0,1109.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,"2/ Hard Red Winter, Hard Red Spring, Soft Red ...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
288,3/ Includes flour and selected other products ...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
289,4/ Totals may not add due to rounding.,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [6]:
import bigframes.pandas as bpd

# TODO(developer): Follow the instructions at
# https://docs.cloud.google.com/bigquery/docs/sandbox and set the project to the
# ID of the project you created.
bpd.options.bigquery.project = "swena-bq-sandbox"

In [7]:
%%bqsql df_with_year
SELECT
  REGEXP_EXTRACT(LTRIM(`Marketing year 1`), r'^([0-9]+)/[0-9]') AS `Start year`,
  SAFE_CAST(`Seed use` AS FLOAT64) AS `Seed use`,
  * EXCEPT (`Marketing year 1`, `Seed use`)
FROM {df_renamed}
ORDER BY rowindex ASC

UsageError: Cell magic `%%bqsql` not found.


In [ ]:
%%bqsql use_proportions
SELECT
  `rowindex`,
  `Start year`,
  `Seed use` / `Total disappearance 4` AS `Seed proportion`,
  `Food use` / `Total disappearance 4` AS `Food proportion`,
  `Feed and residual use` / `Total disappearance 4` AS `Feed proportion`,
  `Exports` / `Total disappearance 4` AS `Exports proportion`
FROM {df_with_year}
WHERE TRIM(`Type 2`) = 'All wheat'
ORDER BY `rowindex` ASC;

In [ ]:
use_proportions.set_index('Start year')[['Seed proportion', 'Food proportion', 'Feed proportion', 'Exports proportion']].plot.area(stacked=True, ylim=(0, 1.5), colormap='viridis')

In [ ]:
type(use_proportions)

In [ ]:
pandas_result = use_proportions.to_pandas()
pandas_result